# Silver elhub price area

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import SparkSession
from silver_layer.elhub_price_area import ingest
from silver_layer import cache

BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
SILVER_ROOT = PROJECT_ROOT / "data" / "silver"
print("Project root:", PROJECT_ROOT)
print("Bronze root exists:", BRONZE_ROOT.exists())

Project root: /home/roland/UiS/rewrite/group14
Bronze root exists: True


## Run the pipeline

Adjust `local[8]` for machine.

In [2]:
spark = (
    SparkSession.builder.master("local[8]")
    .appName("DAT535-Silver-Elhub-Price-Area")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
report = ingest(bronze_root=BRONZE_ROOT, silver_root=SILVER_ROOT, spark=spark)
report

26/11/19 13:20:40 WARN Utils: Your hostname, homelab resolves to a loopback address: 127.0.1.1; using 192.168.10.53 instead (on interface lan0)
26/11/19 13:20:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 13:20:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/19 13:20:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Checking Silver cache: /home/roland/UiS/rewrite/group14/data/silver/elhub/price_area_consumption
Extracted 1 CSV file(s). Processing with Spark


Writing 438,600 rows to staging directory


26/11/19 13:20:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


{
  "dataset": "CONSUMPTION_PER_GROUP_MBA_HOUR",
  "status": "success",
  "rows": 438600,
  "csv_files": 1,
  "processing_seconds": 6.23,
  "silver_path": "/home/roland/UiS/rewrite/group14/data/silver/elhub/price_area_consumption",
  "bronze_archive": "/home/roland/UiS/rewrite/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MBA_HOUR/CONSUMPTION_PER_GROUP_MBA_HOUR_ea329c926f95c9b6.zip"
}


{'dataset': 'CONSUMPTION_PER_GROUP_MBA_HOUR',
 'status': 'success',
 'rows': 438600,
 'csv_files': 1,
 'processing_seconds': 6.23,
 'silver_path': '/home/roland/UiS/rewrite/group14/data/silver/elhub/price_area_consumption',
 'bronze_archive': '/home/roland/UiS/rewrite/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MBA_HOUR/CONSUMPTION_PER_GROUP_MBA_HOUR_ea329c926f95c9b6.zip'}

## Inspect Silver output

In [3]:
output = SILVER_ROOT / "elhub" / "price_area_consumption"
print(cache.read_manifest(output))
df = spark.read.parquet(str(output))
df.printSchema()
df.show(10, truncate=False)

{'bronze_archive': '/home/roland/UiS/rewrite/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MBA_HOUR/CONSUMPTION_PER_GROUP_MBA_HOUR_ea329c926f95c9b6.zip', 'completed_at_utc': '2026-11-19T13:20:47.763337+00:00', 'csv_files': 1, 'dataset': 'CONSUMPTION_PER_GROUP_MBA_HOUR', 'fingerprint': {'end_exclusive_local': '2026-01-01', 'source_sha256': 'ea329c926f95c9b61bf0ddf62e49fbeaecc813e7d57f7bfe4f8e3996a60d6012', 'start_local': '2024-01-01', 'transformation_version': 'price-area-v1'}, 'processing_seconds': 6.23, 'rows': 438600, 'run_id': None}
root
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- price_area: string (nullable = true)
 |-- eic: string (nullable = true)
 |-- consumption_group: string (nullable = true)
 |-- quantity_kwh: decimal(18,6) (nullable = true)
 |-- metering_point_count: integer (nullable = true)
 |-- last_update_time: timestamp (nullable = true)

+-------------------+-------------------+----------+----------------+------------

In [4]:
from pyspark.sql import functions as F

print("Rows:", df.count())
(
    df.groupBy("price_area", "consumption_group")
    .count()
    .orderBy("price_area", "consumption_group")
    .show(30, truncate=False)
)
df.select(
    F.min("start_time").alias("first_utc"),
    F.max("start_time").alias("last_utc"),
).show(truncate=False)
(
    df.groupBy("price_area")
    .agg(F.countDistinct("eic").alias("distinct_eics"))
    .orderBy("price_area")
    .show()
)
df.groupBy("consumption_group").agg(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("quantity_kwh") < 0, 1).otherwise(0)).alias("negative_rows"),
    F.min("quantity_kwh").alias("min_kwh"),
    F.max("quantity_kwh").alias("max_kwh"),
).show(truncate=False)

Rows: 438600
+----------+-----------------+-----+
|price_area|consumption_group|count|
+----------+-----------------+-----+
|NO1       |cabin            |17544|
|NO1       |household        |17544|
|NO1       |primary          |17544|
|NO1       |secondary        |17544|
|NO1       |tertiary         |17544|
|NO2       |cabin            |17544|
|NO2       |household        |17544|
|NO2       |primary          |17544|
|NO2       |secondary        |17544|
|NO2       |tertiary         |17544|
|NO3       |cabin            |17544|
|NO3       |household        |17544|
|NO3       |primary          |17544|
|NO3       |secondary        |17544|
|NO3       |tertiary         |17544|
|NO4       |cabin            |17544|
|NO4       |household        |17544|
|NO4       |primary          |17544|
|NO4       |secondary        |17544|
|NO4       |tertiary         |17544|
|NO5       |cabin            |17544|
|NO5       |household        |17544|
|NO5       |primary          |17544|
|NO5       |secondary    

## Check that no duplicates exist

In [5]:
duplicates = (
    df.groupBy(
        "start_time",
        "price_area",
        "consumption_group",
        "eic",
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate keys:", duplicates.count())

Duplicate keys: 0


## Confirm cache reuse

In [6]:
second_run = ingest(bronze_root=BRONZE_ROOT, silver_root=SILVER_ROOT, spark=spark)
second_run

Checking Silver cache: /home/roland/UiS/rewrite/group14/data/silver/elhub/price_area_consumption
{
  "dataset": "CONSUMPTION_PER_GROUP_MBA_HOUR",
  "status": "cached",
  "rows": 438600,
  "silver_path": "/home/roland/UiS/rewrite/group14/data/silver/elhub/price_area_consumption",
  "bronze_archive": "/home/roland/UiS/rewrite/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MBA_HOUR/CONSUMPTION_PER_GROUP_MBA_HOUR_ea329c926f95c9b6.zip"
}


{'dataset': 'CONSUMPTION_PER_GROUP_MBA_HOUR',
 'status': 'cached',
 'rows': 438600,
 'silver_path': '/home/roland/UiS/rewrite/group14/data/silver/elhub/price_area_consumption',
 'bronze_archive': '/home/roland/UiS/rewrite/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MBA_HOUR/CONSUMPTION_PER_GROUP_MBA_HOUR_ea329c926f95c9b6.zip'}

## Cleanup

In [7]:
spark.stop()